In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from noaa_coops import Station

from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

import utide

In [3]:
# Connect to NOAA station (duck, NC)
duck = Station(id="8651370")

# 2nd period
df = duck.get_data(
    begin_date="20110101",
    end_date="20251231",
    product="hourly_height",
    datum="NAVD",
    units="metric",
    time_zone="gmt"
)

print(f"Date range: {df.index.min()} to {df.index.max()}")
print(f"Total records: {len(df)}")

# The water level data is in column 'v'
df['water_level'] = pd.to_numeric(df['v'], errors='coerce')

# Fill small gaps
df['water_level'] = df['water_level'].interpolate(method='linear', limit=3)

print(f"Missing values after interpolation: {df['water_level'].isna().sum()}")

# Remove missing values
valid_mask = ~np.isnan(df['water_level'])
valid_time = df.index[valid_mask]
valid_water = df['water_level'][valid_mask].values

# Solve for tidal constituents
coef = utide.solve(
    valid_time,
    valid_water,
    lat=36.1833,          # duck, NC latitude
    method='ols',
    conf_int='none',
    nodal=True,
    verbose=False
)

# Tidal predictions
tide = utide.reconstruct(valid_time, coef, verbose=False)
df_tide = pd.Series(tide['h'], index=valid_time, name='tidal_prediction')

# Calculate the non-tidal residual
df_residual = valid_water - tide['h']
df_residual = pd.Series(df_residual, index=valid_time, name='non_tidal_residual')

# Create plots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8), sharex=True)

# Tidal prediction (nA)
ax1.plot(df_tide.index, df_tide, 'r-', linewidth=0.8, label='Tidal Prediction (ηA)')
# ax1.plot(df_tide.index, valid_water, 'b-', linewidth=0.5, alpha=0.5, label='Observed')
ax1.set_ylabel('Water Level (m NAVD88)')
ax1.set_title('Tidal Prediction (ηA) for duck, NC (2011_2025)')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Non-tidal residual (nNTR)
ax2.plot(df_residual.index, df_residual, 'g-', linewidth=0.5)
ax2.axhline(y=0, color='k', linestyle='-', linewidth=0.5)
ax2.set_ylabel('Non-Tidal Residual (m)')
ax2.set_xlabel('Date')
ax2.set_title('Non-Tidal Residual (ηNTR = Observed - Tidal Prediction)')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Create a combined dataframe
results_df = pd.DataFrame({
    'datetime': df_tide.index,
    'tidal_prediction_etaA': df_tide.values,
    'non_tidal_residual_etaNTR': df_residual.values,
    'observed_water_level': valid_water
})

In [4]:
wis_filename = '/Users/rsahrae/PycharmProjects/PeaIsland_Hindcast/CASCADE/data/PeaIsland_init/Storms/2011_2025/WIS_2011_2025.txt'
beach_slope = 0.004
berm_crest = 1.7
g = 9.81

def load_wis_data_correct(filename):
    
    df_wis = pd.read_csv(filename, sep=r'\s+', header=None)
    
    print(f"DataFrame shape: {df_wis.shape}")
    print(f"Number of columns: {len(df_wis.columns)}")
    
    start_date = pd.Timestamp('2011-01-01 00:00:00')
    df_wis['datetime'] = [start_date + timedelta(hours=i) for i in range(len(df_wis))]
    df_wis.set_index('datetime', inplace=True)
    
    # Remove the first datetime column
    df_wis.drop(0, axis=1, inplace=True)
    
    print(f"Date range: {df_wis.index.min()} to {df_wis.index.max()}")
    print(f"Records: {len(df_wis)}")
    
    # Calculate statistics for each column
    col_stats = []
    for col in df_wis.columns:
        col_data = df_wis[col].dropna()
        if len(col_data) > 0:
            mean_val = col_data.mean()
            max_val = col_data.max()
            min_val = col_data.min()
            std_val = col_data.std()
            col_stats.append({
                'col': col,
                'mean': mean_val,
                'max': max_val,
                'min': min_val,
                'std': std_val
            })
    
    # Wave height
    hs_candidates = []
    for stat in col_stats:
        if 0.1 < stat['mean'] < 5 and 0 < stat['min'] and stat['max'] < 15:
            hs_candidates.append(stat)
    
    # Wave period
    tp_candidates = []
    for stat in col_stats:
        if 2 < stat['mean'] < 15 and 0 < stat['min'] and stat['max'] < 30:
            tp_candidates.append(stat)
    
    if hs_candidates:
        hs_candidates.sort(key=lambda x: abs(x['mean'] - 1.5))
        hs_col = hs_candidates[0]['col']
        print(f"\nIdentified wave height column: {hs_col} (mean={hs_candidates[0]['mean']:.2f} m)")
        df_wis['Hs'] = df_wis[hs_col]
    else:
        if 9 in df_wis.columns:
            print(f"\nUsing column 9 for wave height (mean={df_wis[9].mean():.2f} m)")
            df_wis['Hs'] = df_wis[9]
        else:
            print("Could not identify wave height column")
    
    if tp_candidates:
        tp_candidates.sort(key=lambda x: abs(x['mean'] - 8))
        tp_col = tp_candidates[0]['col']
        print(f"Identified wave period column: {tp_col} (mean={tp_candidates[0]['mean']:.2f} s)")
        df_wis['Tp'] = df_wis[tp_col]
    else:
        if 11 in df_wis.columns:
            print(f"Using column 11 for wave period (mean={df_wis[11].mean():.2f} s)")
            df_wis['Tp'] = df_wis[11]
        else:
            print("Could not identify wave period column")
    
    # Wave direction
    dir_candidates = []
    for stat in col_stats:
        if 0 < stat['mean'] < 360 and stat['max'] < 360:
            dir_candidates.append(stat)
    
    if dir_candidates:
        dir_col = dir_candidates[0]['col']
        print(f"Identified wave direction column: {dir_col} (mean={dir_candidates[0]['mean']:.1f}°)")
        df_wis['WAVD'] = df_wis[dir_col]
    elif 15 in df_wis.columns:
        df_wis['WAVD'] = df_wis[15]
    
    # Remove negative values
    if 'Hs' in df_wis.columns:
        df_wis['Hs'] = df_wis['Hs'].clip(lower=0)
    if 'Tp' in df_wis.columns:
        df_wis['Tp'] = df_wis['Tp'].clip(lower=0)
    
    print(f"\nWIS data loaded successfully!")
    print(f"Final columns: {df_wis.columns.tolist()}")
    
    return df_wis

In [5]:
# Load data from part A
# Set datetime index for tide data
if 'datetime' in results_df.columns:
    results_df['datetime'] = pd.to_datetime(results_df['datetime'])
    results_df.set_index('datetime', inplace=True)

# Load WIS data
df_wis = load_wis_data_correct(wis_filename)

# Create a merged dataframe with WIS timestamps
df_merged = pd.DataFrame(index=df_wis.index)

# Add WIS wave data
df_merged['Hs'] = df_wis['Hs']
df_merged['Tp'] = df_wis['Tp']
df_merged['WAVD'] = df_wis['WAVD']

# Add water level from tide data
if 'observed_water_level' in results_df.columns:
    water_level_col = 'observed_water_level'
elif 'water_level_clean' in results_df.columns:
    water_level_col = 'water_level_clean'
else:
    water_level_col = 'v'

# Align tide data to WIS timestamps
df_merged['water_level'] = results_df[water_level_col].reindex(df_wis.index).interpolate(
    method='linear', limit=24, limit_area='inside'
)

# Remove rows with missing data
df_merged = df_merged.dropna(subset=['Hs', 'Tp', 'water_level'])

print(f"Merged data shape: {df_merged.shape}")
print(f"Date range: {df_merged.index.min()} to {df_merged.index.max()}")
print(f"Wave height range: {df_merged['Hs'].min():.2f} to {df_merged['Hs'].max():.2f} m")
print(f"Wave period range: {df_merged['Tp'].min():.2f} to {df_merged['Tp'].max():.2f} s")

In [6]:
def calculate_r2_percent(Hs, Tp, slope):
    """Calculate R2% (2% exceedance runup) using Stockdon et al. (2006)"""
    # Deep water wavelength
    L0 = (g * Tp**2) / (2 * np.pi)
    
    # Setup component
    setup = 0.35 * slope * np.sqrt(Hs * L0)
    
    # Swash components
    S_incident = 0.75 * slope * np.sqrt(Hs * L0)
    S_infragravity = 0.06 * np.sqrt(Hs * L0)
    S_total = np.sqrt(S_incident**2 + S_infragravity**2)
    
    # R2% runup
    R2 = 1.1 * (setup + S_total/2)
    
    return R2
    
df_merged['R2'] = calculate_r2_percent(df_merged['Hs'], df_merged['Tp'], beach_slope)
df_merged['TWL'] = df_merged['water_level'] + df_merged['R2']

In [7]:
fig, axes = plt.subplots(4, 1, figsize=(15, 14))

# Plot Hs
axes[0].plot(df_merged.index, df_merged['Hs'], 'b-', linewidth=0.5, alpha=0.7)
axes[0].set_ylabel('Significant Wave Height (m)')
axes[0].set_title(f'Significant Wave Height (WIS Data) ({df_merged.index.min().year}-{df_merged.index.max().year})')
axes[0].grid(True, alpha=0.3)
if len(df_merged) > 0:
    axes[0].set_ylim([0, df_merged['Hs'].quantile(0.99)])

# Plot Tp
axes[1].plot(df_merged.index, df_merged['Tp'], 'r-', linewidth=0.5, alpha=0.7)
axes[1].set_ylabel('Peak Period (s)')
axes[1].set_title('Wave Period (Tp)')
axes[1].grid(True, alpha=0.3)
if len(df_merged) > 0:
    axes[1].set_ylim([0, df_merged['Tp'].quantile(0.99)])

# Plot R2%
axes[2].plot(df_merged.index, df_merged['R2'], 'g-', linewidth=0.5, alpha=0.7)
axes[2].set_ylabel('R2% Runup (m)')
axes[2].set_title('2% Exceedance Runup (Stockdon et al. 2006)')
axes[2].grid(True, alpha=0.3)

# Plot Water Level and TWL
axes[3].plot(df_merged.index, df_merged['water_level'], 'b-', linewidth=0.5, alpha=0.5, label='Water Level')
axes[3].plot(df_merged.index, df_merged['TWL'], 'k-', linewidth=0.5, alpha=0.7, label='Total Water Level (TWL)')
axes[3].axhline(y=berm_crest, color='r', linestyle='--', linewidth=2, 
                label=f'Berm Crest = {berm_crest} m NAVD88')
axes[3].set_ylabel('Elevation (m NAVD88)')
axes[3].set_title('Water Level and Total Water Level')
axes[3].legend(loc='upper right')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [27]:
import pandas as pd
import numpy as np

# =========================
# USER INPUT
# =========================
berm_elevation = 1.7
tp_col = "Tp"   # change if your WIS period column has a different name
min_storm_duration_hours = 6   # keep only storms longer than 6 hr

# =========================
# BUILD DATAFRAME
# =========================
df = pd.DataFrame({
    "Time": pd.to_datetime(df_merged.index),
    "TWL": pd.to_numeric(df_merged["TWL"], errors="coerce")
})

# Add Tp safely
if tp_col in df_merged.columns:
    df["Tp"] = pd.to_numeric(df_merged[tp_col], errors="coerce")
else:
    df["Tp"] = np.nan

df = df.sort_values("Time").reset_index(drop=True)

# =========================
# CLEAN / FILL SMALL Tp GAPS
# =========================
# Fill small internal gaps first
df["Tp"] = df["Tp"].interpolate(limit=6)

# Then forward/backward fill any remaining gaps
df["Tp"] = df["Tp"].ffill().bfill()

# Optional debug
print("Total rows:", len(df))
print("Non-NaN TWL:", df["TWL"].notna().sum())
print("Non-NaN Tp:", df["Tp"].notna().sum())
print("NaN Tp:", df["Tp"].isna().sum())
# =========================
if len(df) > 1:
    dt_hours_global = df["Time"].diff().dt.total_seconds().median() / 3600.0
else:
    dt_hours_global = 1.0

print("Detected time step:", dt_hours_global, "hours")

# =========================
# FIND STORMS (TWL > berm)
# =========================
df["AboveBerm"] = df["TWL"] > berm_elevation

df["StormStart"] = df["AboveBerm"] & (~df["AboveBerm"].shift(1, fill_value=False))

df["StormID"] = df["StormStart"].cumsum()
df.loc[~df["AboveBerm"], "StormID"] = np.nan

# =========================
# BUILD STORMS
# =========================
storms = []

storm_groups = df.dropna(subset=["StormID"]).groupby("StormID")

for sid, group in storm_groups:

    group = group.sort_values("Time").copy()

    start_time = group["Time"].iloc[0]
    end_time   = group["Time"].iloc[-1]

    # Duration in hours
    # Add one time step because each row represents one interval
    if len(group) > 1:
        dt_hours = (
            group["Time"].iloc[1] - group["Time"].iloc[0]
        ).total_seconds() / 3600.0

        duration = (
            (end_time - start_time).total_seconds() / 3600.0
        ) + dt_hours

    else:
        duration = dt_hours_global

    # Keep only storms longer than 6 hours
    if duration > min_storm_duration_hours:
        continue
    # Rhigh and Rlow from TWL during the storm
    rhigh = group["TWL"].max()
    rlow  = group["TWL"].min()
    #convert to dam
    rhigh = rhigh/10
    rlow = rlow/10
    # Period from Tp at PEAK TWL
    if group["Tp"].notna().any():
        peak_idx = group["TWL"].idxmax()
        period = group.loc[peak_idx, "Tp"]
    else:
        # fallback in case Tp is still missing
        period = df["Tp"].mean()

    storms.append({
        "calendar_year": start_time.year,
        "StartTime": start_time,
        "EndTime": end_time,
        "Rhigh": rhigh,
        "Rlow": rlow,
        "Period": period,
        "duration": duration
    })

storms_df = pd.DataFrame(storms)

# =========================
# CONVERT YEAR → TIME (CASCADE)
# =========================
if not storms_df.empty:
    start_year = storms_df["calendar_year"].min()
    storms_df["time"] = storms_df["calendar_year"] - start_year + 1
else:
    storms_df["time"] = pd.Series(dtype=float)

# =========================
# FINAL FORMAT
# =========================
cascade_df = storms_df[["time", "Rhigh", "Rlow", "Period", "duration"]].copy()
cascade_df = cascade_df.reset_index(drop=True)

# =========================
# OUTPUT
# =========================
print("\nStorm summary:")
print(storms_df.head())

print("\nCASCADE format:")
print(cascade_df.head())
storms_df.to_csv("storm_summary_2011_2025.csv", index=True)
cascade_df.to_csv("cascade_storms_2011_2025.csv", index=True)

In [28]:
cascade_df

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# =========================
# USER INPUT
# =========================
storm_file = Path(
    "/Users/rsahrae/PycharmProjects/PeaIsland_Hindcast/CASCADE/notebooks/cascade_storms_2011_2025.csv"
)

save_dir = Path(
    "/Users/rsahrae/PycharmProjects/PeaIsland_Hindcast/CASCADE/data/PeaIsland_init/Storms/2011_2025"
)

berm_elevation = 1.7
start_year = 2011   # change to 1992 if plotting 1992_2010 file

SHOW_PLOTS = True
SAVE_PLOTS = True

# =========================
# LOAD EXISTING STORM FILE
# =========================
storms = pd.read_csv(storm_file)

# Clean possible unnamed index column
storms = storms.loc[:, ~storms.columns.str.contains("^Unnamed")]

print("Loaded storm file:")
print(storm_file)
print("Columns:", storms.columns.tolist())
print("Number of storms:", len(storms))
print(storms.head())

# =========================
# BUILD TWL DATAFRAME FOR PLOT
# =========================
df_plot = pd.DataFrame({
    "Time": pd.to_datetime(df_merged.index),
    "TWL": pd.to_numeric(df_merged["TWL"], errors="coerce")
})

df_plot = df_plot.dropna(subset=["TWL"])
df_plot = df_plot.sort_values("Time")
df_plot = df_plot.set_index("Time")

# =========================
# CONVERT CASCADE TIME TO DATETIME
# =========================
def model_year_to_datetime(model_year, start_year):
    """
    Convert CASCADE model time in years to approximate calendar datetime.

    Example:
    start_year = 2011
    time = 0.0 -> Jan 1 2011
    time = 1.0 -> Jan 1 2012
    time = 0.5 -> about mid-2011
    """
    base = pd.Timestamp(year=start_year, month=1, day=1)
    return base + pd.to_timedelta(model_year * 365.25, unit="D")


storms["StartTime"] = storms["time"].apply(
    lambda x: model_year_to_datetime(x, start_year)
)

storms["EndTime"] = storms["StartTime"] + pd.to_timedelta(
    storms["duration"], unit="h"
)

events = list(zip(storms["StartTime"], storms["EndTime"]))

# =========================
# PLOT 1: TWL TIME SERIES WITH STORMS SHADED
# =========================
fig, ax = plt.subplots(figsize=(18, 5))

ax.plot(
    df_plot.index,
    df_plot["TWL"],
    color="steelblue",
    lw=0.5,
    alpha=0.8,
    label="Total Water Level = water level + R2%",
)

ax.axhline(
    berm_elevation,
    color="red",
    ls="--",
    lw=1.5,
    label=f"Berm elevation ({berm_elevation} m)",
)

for s, e in events:
    ax.axvspan(s, e, alpha=0.25, color="red")

ax.set_ylabel("Elevation (m NAVD88)")
ax.set_xlabel("Date")
ax.set_title("Existing CASCADE Storm File Plotted on TWL Time Series")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()

if SAVE_PLOTS:
    save_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        save_dir / "existing_storm_file_on_TWL.png",
        dpi=300,
        bbox_inches="tight",
    )

if SHOW_PLOTS:
    plt.show()
else:
    plt.close()

# =========================
# PLOT 2: STORM PARAMETER DISTRIBUTIONS
# =========================
if len(storms) > 0:

    fig, axs = plt.subplots(2, 2, figsize=(14, 10))

    fig.suptitle(
        f"Existing Storm File Parameter Distributions "
        f"(N={len(storms)}, start year={start_year})",
        fontsize=14,
    )

    storms["Rhigh"].hist(
        ax=axs[0, 0],
        bins=20,
        color="#1f77b4",
        edgecolor="k",
        alpha=0.8,
    )
    axs[0, 0].set(
        title="Rhigh Distribution",
        xlabel="Rhigh",
        ylabel="Count",
    )

    storms["Period"].hist(
        ax=axs[0, 1],
        bins=20,
        color="#ff7f0e",
        edgecolor="k",
        alpha=0.8,
    )
    axs[0, 1].set(
        title="Wave Period Distribution",
        xlabel="Period (s)",
        ylabel="Count",
    )

    storms["duration"].hist(
        ax=axs[1, 0],
        bins=20,
        color="#2ca02c",
        edgecolor="k",
        alpha=0.8,
    )
    axs[1, 0].set(
        title="Storm Duration Distribution",
        xlabel="Duration (hours)",
        ylabel="Count",
    )

    axs[1, 1].scatter(
        storms["Rlow"],
        storms["Rhigh"],
        alpha=0.7,
        color="#d62728",
        s=30,
    )
    axs[1, 1].set(
        title="Rlow vs Rhigh",
        xlabel="Rlow",
        ylabel="Rhigh",
    )

    for ax in axs.flat:
        ax.grid(alpha=0.3)

    plt.tight_layout(rect=[0, 0, 1, 0.96])

    if SAVE_PLOTS:
        fig.savefig(
            save_dir / "existing_storm_file_distributions.png",
            dpi=300,
            bbox_inches="tight",
        )

    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close()

else:
    print("Storm file is empty. No distribution plot created.")